# NumCompute quickstart

End-to-end demo of the `numcompute` package.

In [1]:
%pip install -e ..

Obtaining file:///Users/danushijayasekara/Developer/AU/Progamming%20for%20AI/Code_Cube-COMP-5004-Assignment-2.1-Programming-Task-1
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for numcompute (pyproject.toml) ... done
  Created wheel for numcompute: filename=numcompute-0.1.0-0.editable-py3-none-any.whl size=3817 sha256=3348e55929f5c0bd3ed0aefd8211563c93a0667b1cb25a961d0b90cd831cb958
  Stored in directory: /private/var/folders/lm/rsky81396zz3gxdz_cr8x5c80000gn/T/pip-ephem-wheel-cache-r3kjoyde/wheels/19/86/33/f7e9a1fe7bec4996a8485092a17af07fff1419a914671416eb
Successfully built numcompute
  Attempting uninstall: numcompute
    Found existing installation: numcompute 0.1.0
    Uninstalling numcompute-0.1.0:
      Successfully uninstalled numcompute-0.1.0
Note: you may need to restart the kernel to use updated 

In [2]:
import numpy as np
import numcompute

## Load, inspect, clean and prepare the data

This part of the demo uses three modules: `io` (loading), `utils`
(helpers) and `preprocessing` (cleaning). 

In [ ]:
from numcompute import io as nc_io, stats
from numcompute.preprocessing import (
    StandardScaler, SimpleImputer, OneHotEncoder)

path = "data/penguins.csv"
names = nc_io.read_header(path)
for i, name in enumerate(names):
    print(i, name)

0 species
1 island
2 bill_length_mm
3 bill_depth_mm
4 flipper_length_mm
5 body_mass_g
6 sex


### Load the file
`load_mixed_csv` reads every value as text first, then converts. It returns
two arrays: 
- `X_num` : (numbers, with NaN where a value was missing) 
- `X_cat` : (text categories).

In [4]:
num_cols = [2, 3, 4, 5]    
cat_cols = [0, 1, 6] 

X_num, X_cat = nc_io.load_mixed_csv(path, num_cols, cat_cols, skip_header=1)

num_names = [names[i] for i in num_cols]
cat_names = [names[i] for i in cat_cols]

print("numeric    :", num_names, X_num.shape, X_num.dtype)
print("categorical:", cat_names, X_cat.shape, X_cat.dtype)

numeric    : ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g'] (344, 4) float64
categorical: ['species', 'island', 'sex'] (344, 3) <U9


In [5]:
print(f"{'':>3}" + "".join(f"{n[:15]:>17}" for n in num_names)
      + "".join(f"{n:>12}" for n in cat_names))
for i in range(5):
    print(f"{i:>3}" + "".join(f"{v:>17}" for v in X_num[i])
          + "".join(f"{v:>12}" for v in X_cat[i]))

      bill_length_mm    bill_depth_mm  flipper_length_      body_mass_g     species      island         sex
  0             39.1             18.7            181.0           3750.0      Adelie   Torgersen        MALE
  1             39.5             17.4            186.0           3800.0      Adelie   Torgersen      FEMALE
  2             40.3             18.0            195.0           3250.0      Adelie   Torgersen      FEMALE
  3              nan              nan              nan              nan      Adelie   Torgersen            
  4             36.7             19.3            193.0           3450.0      Adelie   Torgersen      FEMALE


### Missing data
Missing numbers must be filled before scaling.

In [6]:
missing = np.isnan(X_num)
print("missing per column:", dict(zip(num_names, missing.sum(axis=0).tolist())))
rows_with_gaps = np.flatnonzero(missing.any(axis=1))
print("rows with a missing number:", rows_with_gaps.size, "of", X_num.shape[0])
print("row numbers:", rows_with_gaps.tolist())

missing per column: {'bill_length_mm': 2, 'bill_depth_mm': 2, 'flipper_length_mm': 2, 'body_mass_g': 2}
rows with a missing number: 2 of 344
row numbers: [3, 339]


### Summary of the numbers and the categories
For each measurement: mean, median, spread (standard deviation), smallest and
largest value, calculated with `numcompute.stats` while ignoring `nan`.

In [7]:
print(f"{'column':<20}{'mean':>10}{'median':>10}{'std':>10}"
      f"{'min':>10}{'max':>10}")
for j, name in enumerate(num_names):
    col = X_num[:, j]
    print(f"{name:<20}"
          f"{stats.mean(col, nan_policy='omit'):>10.1f}"
          f"{stats.median(col, nan_policy='omit'):>10.1f}"
          f"{stats.std(col, nan_policy='omit'):>10.1f}"
          f"{stats.min(col, nan_policy='omit'):>10.1f}"
          f"{stats.max(col, nan_policy='omit'):>10.1f}")

column                    mean    median       std       min       max
bill_length_mm            43.9      44.5       5.5      32.1      59.6
bill_depth_mm             17.2      17.3       2.0      13.1      21.5
flipper_length_mm        200.9     197.0      14.0     172.0     231.0
body_mass_g             4201.8    4050.0     800.8    2700.0    6300.0


In [8]:
for j, name in enumerate(cat_names):
    values, counts = np.unique(X_cat[:, j], return_counts=True)
    print(name, dict(zip(values.tolist(), counts.tolist())))

species {'Adelie': 152, 'Chinstrap': 68, 'Gentoo': 124}
island {'Biscoe': 168, 'Dream': 124, 'Torgersen': 52}
sex {'': 11, 'FEMALE': 165, 'MALE': 168}


### Fill missing numbers.

`SimpleImputer` replaces each missing number
with the median of its column. The median is a good choice because
extreme values do not affect it much.

In [9]:
first_gap = int(rows_with_gaps[0])
imputer = SimpleImputer("median").fit(X_num)
filled = imputer.transform(X_num)

print("missing numbers left:", int(np.isnan(filled).sum()))
print("medians used:", dict(zip(num_names, imputer.statistics_.round(1).tolist())))
print(f"row {first_gap} before:", X_num[first_gap])
print(f"row {first_gap} after :", filled[first_gap])

missing numbers left: 0
medians used: {'bill_length_mm': 44.4, 'bill_depth_mm': 17.3, 'flipper_length_mm': 197.0, 'body_mass_g': 4050.0}
row 3 before: [nan nan nan nan]
row 3 after : [  44.45   17.3   197.   4050.  ]


### Scale the numbers.

`StandardScaler` rescales each column so it has
mean 0 and standard deviation 1.

In [10]:
scaler = StandardScaler().fit(filled)
Z = scaler.transform(filled)

print("mean per column:", Z.mean(axis=0).round(3))
print("std per column :", Z.std(axis=0).round(3))
print("row 0 raw   :", filled[0])
print("row 0 scaled:", Z[0].round(2))
print("undo scaling works:", np.allclose(scaler.inverse_transform(Z), filled))

mean per column: [-0. -0.  0.  0.]
std per column : [1. 1. 1. 1.]
row 0 raw   : [  39.1   18.7  181.  3750. ]
row 0 scaled: [-0.89  0.79 -1.42 -0.56]
undo scaling works: True


### Encode the categories.
`OneHotEncoder` gives every category its own 0/1 column.

In [11]:
enc = OneHotEncoder().fit(X_cat)
H = enc.transform(X_cat)

for name, cats in zip(cat_names, enc.categories_):
    print(name, "->", cats.tolist())
print("encoded shape:", H.shape)
print("row 0:", X_cat[0].tolist(), "->", H[0])
print("every row has one 1 per column:",
      bool(np.all(H.sum(axis=1) == len(cat_cols))))

species -> ['Adelie', 'Chinstrap', 'Gentoo']
island -> ['Biscoe', 'Dream', 'Torgersen']
sex -> ['', 'FEMALE', 'MALE']
encoded shape: (344, 9)
row 0: ['Adelie', 'Torgersen', 'MALE'] -> [1. 0. 0. 0. 0. 1. 0. 0. 1.]
every row has one 1 per column: True


In [12]:
features = np.hstack([Z, H])
print("feature matrix:", features.shape)

feature matrix: (344, 13)
